# LeetCode 846 — Hand of Straights bằng Python

Link đề: https://leetcode.com/problems/hand-of-straights/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def isNStraightHand(self, hand: List[int], groupSize: int) -> bool:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (`True`/`False`). Đề này **giống hệt** LeetCode 1296 (*Divide Array in Sets of K Consecutive Numbers*). Điểm cốt lõi là **tham lam**: lá bài **nhỏ nhất** còn lại bắt buộc là lá nhỏ nhất của một nhóm. Notebook kiểm chứng tham lam bằng **oracle độc lập**: liệt kê **mọi cách chia** `n` lá bài thành các nhóm `groupSize` rồi kiểm tra nhóm có liên tiếp không.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia `hand` thành các nhóm, mỗi nhóm gồm đúng `groupSize` lá bài có giá trị **liên tiếp** (ví dụ `[3,4,5]`); hỏi có chia được không.
2. Nắm các công cụ Python: `collections.Counter` (đếm tần suất), `sorted` các khoá, `heapq` (min-heap), `deque`, toán tử `%` để kiểm tra chia hết, `itertools.combinations` (liệt kê cách chọn nhóm).
3. Đi từ **lặp: lấy lá nhỏ nhất rồi xoá `groupSize` lá liên tiếp khỏi danh sách (`O(n²)`)** → **`Counter` + duyệt khoá đã sắp, trừ theo số lượng** → **min-heap** → **quét một lượt với số nhóm "đang mở" (sweep)**.
4. Hiểu mấu chốt: lá bài **nhỏ nhất** `m` chỉ có thể là lá nhỏ nhất của nhóm chứa nó, nên nhóm đó **buộc** là `m, m+1, …, m+k−1`. Lấy nhóm này ra và lặp lại: không có lựa chọn nào khác để cân nhắc, nên tham lam **không bao giờ sai**.
5. Tránh các lỗi kinh điển: quên kiểm tra `len(hand) % groupSize`, chỉ kiểm tra **có mặt** (dùng `set`) mà quên **số lượng**, bắt đầu nhóm từ lá **không nhỏ nhất**, sửa `Counter` trong lúc duyệt, trừ sai số lượng, và quên rằng giá trị có thể lớn (`10^9`) nên không dùng mảng đếm theo giá trị.

## 2. Đề bài & ví dụ

Alice có một số lá bài và muốn xếp chúng thành các **nhóm**, mỗi nhóm có đúng `groupSize` lá bài và gồm các giá trị **liên tiếp** (mỗi giá trị hơn giá trị trước đúng `1`).

Cho mảng `hand` (`hand[i]` là giá trị ghi trên lá bài thứ `i`) và số nguyên `groupSize`, trả về `true` nếu Alice sắp xếp được, ngược lại `false`.

**Ví dụ 1:** `hand = [1,2,3,6,2,3,4,7,8]`, `groupSize = 3` → `true`. Chia thành `[1,2,3]`, `[2,3,4]`, `[6,7,8]`.

**Ví dụ 2:** `hand = [1,2,3,4,5]`, `groupSize = 4` → `false`. Có `5` lá không chia hết cho nhóm `4` lá.

**Ràng buộc:**

- `1 <= hand.length <= 10^4`
- `0 <= hand[i] <= 10^9`
- `1 <= groupSize <= hand.length`

**Ghi chú:** đề này giống bài 1296.

In [1]:
import random
import time
from collections import Counter, deque
from heapq import heapify, heappop
from itertools import combinations, combinations_with_replacement
from typing import List

ex1 = ([1, 2, 3, 6, 2, 3, 4, 7, 8], 3)
ex2 = ([1, 2, 3, 4, 5], 4)
for hand, k in (ex1, ex2):
    print(f"hand={hand}, groupSize={k}: n={len(hand)}, n % k = {len(hand) % k}, tần suất = {dict(sorted(Counter(hand).items()))}")

hand=[1, 2, 3, 6, 2, 3, 4, 7, 8], groupSize=3: n=9, n % k = 0, tần suất = {1: 1, 2: 2, 3: 2, 4: 1, 6: 1, 7: 1, 8: 1}
hand=[1, 2, 3, 4, 5], groupSize=4: n=5, n % k = 1, tần suất = {1: 1, 2: 1, 3: 1, 4: 1, 5: 1}


## 3. Kiến thức nền

### 3.1 `Counter`: đếm tần suất

`Counter(iterable)` là từ điển `{giá trị: số lần xuất hiện}`. Truy cập khoá **không có** trả `0` (và **không** thêm khoá vào `Counter`). `sorted(counter)` cho các khoá tăng dần. Dùng `Counter` thay cho danh sách vì mỗi giá trị có thể lặp nhiều lần và giá trị lớn (`10^9`) nên **không** lập mảng đếm theo giá trị.

In [2]:
c = Counter([1, 2, 3, 6, 2, 3, 4, 7, 8])
print(c)
print("c[2] =", c[2], "| c[99] =", c[99], "| 99 có trong c?", 99 in c)
print("khoá tăng dần:", sorted(c))
c[2] -= 1
print("sau khi trừ 1 ở khoá 2:", dict(sorted(c.items())))

Counter({2: 2, 3: 2, 1: 1, 6: 1, 4: 1, 7: 1, 8: 1})
c[2] = 2 | c[99] = 0 | 99 có trong c? False
khoá tăng dần: [1, 2, 3, 4, 6, 7, 8]
sau khi trừ 1 ở khoá 2: {1: 1, 2: 1, 3: 2, 4: 1, 6: 1, 7: 1, 8: 1}


### 3.2 Chia hết và điều kiện cần

Mỗi nhóm có `groupSize` lá, vậy tổng số lá phải là **bội** của `groupSize`: `len(hand) % groupSize == 0`. Nếu không thì trả `False` ngay (điều kiện **cần**, chưa **đủ**).

### 3.3 Nhóm liên tiếp là gì

Nhóm `[x, x+1, …, x+k−1]`. Với giá trị nhỏ nhất `x`, các thành viên còn lại hoàn toàn xác định. Vì vậy chỉ cần quyết định **mỗi nhóm bắt đầu từ giá trị nào** — và với lá nhỏ nhất `m` của cả bộ bài, nhóm chứa nó chỉ có thể **bắt đầu** từ `m` (vì không có lá nào nhỏ hơn `m` để đứng trước nó trong nhóm).

### 3.4 Min-heap

`heapq` biến `list` thành **min-heap**: phần tử nhỏ nhất ở `heap[0]`; `heappop` lấy và bỏ nó trong `O(log n)`. Dùng để luôn lấy được "giá trị nhỏ nhất còn lá bài".

### 3.5 Hàng đợi `deque`

`deque` cho `append` và `popleft` ở `O(1)`; dùng ở cách cuối để nhớ "bao nhiêu nhóm bắt đầu ở mỗi giá trị trong `groupSize` giá trị gần nhất".

In [3]:
h = [5, 1, 3, 2, 4]
heapify(h)
print("đỉnh heap:", h[0], "| lấy lần lượt:", [heappop(h) for _ in range(5)])
q = deque()
for x in (2, 0, 3):
    q.append(x)
print("deque:", q.popleft(), list(q))

đỉnh heap: 1 | lấy lần lượt: [1, 2, 3, 4, 5]
deque: 2 [0, 3]


## 4. Ý tưởng nền tảng / trực giác

Sắp xếp bài tăng dần (ví dụ 1): `1 2 2 3 3 4 6 7 8`. Lá nhỏ nhất là `1`. Nhóm chứa `1` có giá trị nhỏ nhất là `1`, nên **phải** là `[1, 2, 3]`: lấy `1, 2, 3` ra. Còn `2 3 4 6 7 8`: lá nhỏ nhất `2` → nhóm `[2, 3, 4]`. Còn `6 7 8` → nhóm `[6, 7, 8]`. Hết bài → `True`.

Ví dụ lỗi: `[1, 2, 3, 3, 4, 5]`, `k = 3`: lá nhỏ nhất `1` → `[1,2,3]`; còn `3 4 5` → `[3,4,5]` ✓. Ví dụ khác: `[1, 2, 3, 4, 5, 6]`, `k = 4`: `6 % 4 ≠ 0` → `False` ngay. Ví dụ `[1, 1, 2, 2, 3, 4]`, `k = 3`: `[1,2,3]` rồi còn `1, 2, 4`: `1` → cần `2, 3` nhưng không còn `3` → `False`.

Ta không cần thử "có nên bắt đầu nhóm ở đây không": lá nhỏ nhất luôn buộc bắt đầu một nhóm. Mỗi bước loại đúng `k` lá khỏi bộ bài.

In [4]:
def trace_simple(hand, k):
    h = sorted(hand)
    print("bài đã sắp:", h, "| k =", k)
    if len(h) % k:
        print("n không chia hết cho k -> False")
        return False
    while h:
        m = h[0]
        group = list(range(m, m + k))
        try:
            for v in group:
                h.remove(v)
        except ValueError:
            print(f"cần nhóm {group} nhưng thiếu {v} -> False")
            return False
        print(f"lấy nhóm {group}; còn lại {h}")
    print("hết bài -> True")
    return True


print(trace_simple(*ex1))
print(trace_simple([1, 1, 2, 2, 3, 4], 3))

bài đã sắp: [1, 2, 2, 3, 3, 4, 6, 7, 8] | k = 3
lấy nhóm [1, 2, 3]; còn lại [2, 3, 4, 6, 7, 8]
lấy nhóm [2, 3, 4]; còn lại [6, 7, 8]
lấy nhóm [6, 7, 8]; còn lại []
hết bài -> True
True
bài đã sắp: [1, 1, 2, 2, 3, 4] | k = 3
lấy nhóm [1, 2, 3]; còn lại [1, 2, 4]
cần nhóm [1, 2, 3] nhưng thiếu 3 -> False
False


## 5. Cách 1 — Brute Force: lấy lá nhỏ nhất, xoá `groupSize` lá liên tiếp khỏi danh sách

**Ý tưởng:** sắp `hand` tăng dần; trong khi còn bài: lấy giá trị nhỏ nhất `m`, rồi xoá khỏi danh sách một lá `m`, `m+1`, …, `m+k−1` bằng `list.remove` (xoá lần xuất hiện đầu tiên); nếu thiếu lá nào thì `False`.

In [5]:
def hand_remove(hand, k):
    h = sorted(hand)
    if len(h) % k:
        return False
    while h:
        m = h[0]
        for v in range(m, m + k):
            try:
                h.remove(v)                       # O(n): tìm và dịch phần tử
            except ValueError:
                return False
    return True


print(hand_remove(*ex1), hand_remove(*ex2))
assert hand_remove(*ex1) is True and hand_remove(*ex2) is False

True False


**Độ phức tạp:** mỗi nhóm gọi `k` lần `remove` (mỗi lần `O(n)`), có `n/k` nhóm: tổng `O(n²)` (`n = 10^4` → `10^8`, quá chậm trong Python). Bộ nhớ `O(1)` thêm.

**Nút thắt:** `list.remove` tìm và dịch cả mảng. Ta chỉ cần **đếm** số lá mỗi giá trị rồi trừ.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam đúng:** lá nhỏ nhất `m` còn lại phải là lá nhỏ nhất của nhóm chứa nó, nên nhóm đó là `[m, …, m+k−1]` — **không có lựa chọn nào khác**. Lặp lại cho tới hết bài.

Tăng tốc: thay danh sách bằng `Counter` các giá trị. Với giá trị nhỏ nhất `x` có `c = cnt[x] > 0`, ta phải mở **`c` nhóm** bắt đầu từ `x` (mỗi lá `x` là lá nhỏ nhất của một nhóm, vì mọi lá nhỏ hơn đã được dùng hết). Do đó mỗi giá trị `x, x+1, …, x+k−1` phải còn **ít nhất `c` lá**; trừ `c` ở cả `k` giá trị. Làm như vậy cho từng khoá theo thứ tự tăng dần: tổng thời gian `O(m · k)` với `m` là số giá trị khác nhau, hoặc `O(n log n)` nếu dùng thêm kỹ thuật "nhóm đang mở".

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** `Counter` + duyệt khoá đã sắp (trừ theo số lượng), **Cách 3** min-heap, **Cách 4** quét một lượt với số nhóm đang mở.

### Cách 2 — `Counter` + duyệt khoá đã sắp

**Giải thích từng đoạn:**

1. Nếu `len(hand) % k != 0` → `False`.
2. `cnt = Counter(hand)`. Duyệt các khoá `x` theo thứ tự **tăng dần** (danh sách `sorted(cnt)` tính **trước** khi sửa).
3. Với `x` còn `c = cnt[x] > 0`: với mỗi `y` từ `x` đến `x + k − 1`: nếu `cnt[y] < c` → `False`; ngược lại `cnt[y] -= c`.
4. Duyệt xong → `True`.

In [6]:
def hand_counter(hand, k):
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    for x in sorted(cnt):                       # sorted(...) tạo danh sách khoá một lần, an toàn khi sửa cnt
        c = cnt[x]
        if c:
            for y in range(x, x + k):
                if cnt[y] < c:
                    return False
                cnt[y] -= c                     # mở c nhóm bắt đầu từ x
    return True


print(hand_counter(*ex1), hand_counter(*ex2))
assert hand_counter(*ex1) is True and hand_counter(*ex2) is False

True False


In [7]:
# Code nộp LeetCode cho Cách 2 (đáp án chính)
from collections import Counter


class SolutionCounter:
    def isNStraightHand(self, hand: List[int], groupSize: int) -> bool:
        if len(hand) % groupSize:
            return False
        cnt = Counter(hand)
        for x in sorted(cnt):
            c = cnt[x]
            if c:
                for y in range(x, x + groupSize):
                    if cnt[y] < c:
                        return False
                    cnt[y] -= c
        return True

**Độ phức tạp:** sắp xếp các khoá `O(m log m)` (`m` = số giá trị khác nhau). Vòng chính: chỉ những khoá `x` còn `c > 0` lá (tức điểm bắt đầu của các nhóm) mới chạy vòng `for y in range(x, x + k)`; tổng số **nhóm** (tính cả bội `c`) là `n / k`, nhưng mỗi khoá khởi chạy `k` phép trừ bất kể `c`, và mỗi lần trừ tiêu thụ ít nhất một lá bài ở `y = x` rồi các lá còn lại; mỗi lá bài bị trừ đúng một lần theo giá trị nên tổng số phép trừ có ý nghĩa không quá `n` (cộng `O(m)` để duyệt khoá đã hết lá). Tóm lại thời gian `O(n + m log m)`, bộ nhớ `O(m)`.

### Cách 3 — Min-heap các giá trị còn bài

**Giải thích từng đoạn:**

1. `cnt = Counter(hand)`; `heap = list(cnt)`; `heapify(heap)`: các giá trị khác nhau trong heap.
2. Trong khi heap không rỗng: `x = heap[0]` (nhỏ nhất còn bài). Với `y` từ `x` đến `x + k − 1`: nếu `cnt[y] == 0` → `False`; `cnt[y] -= 1`; nếu `cnt[y]` về `0`: giá trị `y` phải đang là **đỉnh heap** (`y == heap[0]`) nếu không thì còn giá trị nhỏ hơn bị bỏ lại ở giữa (không thể) → `False`; ngược lại `heappop`.
3. Heap rỗng → `True`.

Cách này mỗi lần **chỉ tạo một nhóm** (trừ 1) nên số vòng là `n / k`.

In [8]:
def hand_heap(hand, k):
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    heap = list(cnt)
    heapify(heap)
    while heap:
        x = heap[0]
        for y in range(x, x + k):
            if cnt[y] == 0:
                return False
            cnt[y] -= 1
            if cnt[y] == 0:
                if y != heap[0]:                # y hết bài nhưng còn giá trị nhỏ hơn y chưa hết: không thể
                    return False
                heappop(heap)
    return True


print(hand_heap(*ex1), hand_heap(*ex2))
assert hand_heap(*ex1) is True and hand_heap(*ex2) is False

True False


In [9]:
# Code nộp LeetCode cho Cách 3
from heapq import heapify, heappop


class SolutionHeap:
    def isNStraightHand(self, hand: List[int], groupSize: int) -> bool:
        if len(hand) % groupSize:
            return False
        cnt = Counter(hand)
        heap = list(cnt)
        heapify(heap)
        while heap:
            x = heap[0]
            for y in range(x, x + groupSize):
                if cnt[y] == 0:
                    return False
                cnt[y] -= 1
                if cnt[y] == 0:
                    if y != heap[0]:
                        return False
                    heappop(heap)
        return True

**Độ phức tạp:** `n / k` nhóm × `k` bước = `O(n)` thao tác `Counter` cộng `O(m log m)` cho heap: `O(n + m log m)`. Bộ nhớ `O(m)`.

### Cách 4 — Quét một lượt với số nhóm "đang mở"

**Ý tưởng khác:** duyệt các giá trị khác nhau theo thứ tự tăng dần và chỉ nhớ **bao nhiêu nhóm đang mở** (đã bắt đầu nhưng chưa đủ `k` lá). Gọi `opened` là số nhóm đang chờ lá ở giá trị hiện tại. Với giá trị `x` có `cnt[x]` lá:

- Nếu `opened > 0` mà `x ≠ last + 1` (đứt quãng) → các nhóm đang mở không thể nối tiếp → `False`.
- Nếu `cnt[x] < opened` (không đủ lá cho các nhóm đang mở) → `False`.
- Số nhóm **mới** bắt đầu tại `x` là `cnt[x] − opened`; ghi vào hàng đợi `q`. Số nhóm đang mở sau khi dùng `x` là `cnt[x]` (mọi lá `x` đều thuộc nhóm đang mở hoặc mới).
- Khi hàng đợi đủ `k` phần tử (nhóm bắt đầu cách đây `k` giá trị đã đủ `k` lá): bỏ `q.popleft()` nhóm đó khỏi `opened`.
- Cuối cùng cần `opened == 0`.

In [10]:
def hand_sweep(hand, k):
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    q = deque()                                   # q[i]: số nhóm bắt đầu ở giá trị thứ i trong cửa sổ k giá trị gần nhất
    last, opened = -1, 0
    for x in sorted(cnt):
        if opened > 0 and x != last + 1:          # đứt quãng: nhóm đang mở không nối tiếp được
            return False
        if cnt[x] < opened:                       # không đủ lá cho các nhóm đang mở
            return False
        q.append(cnt[x] - opened)                 # nhóm mới bắt đầu tại x
        last, opened = x, cnt[x]                  # mọi lá x đều nằm trong nhóm đang mở
        if len(q) == k:
            opened -= q.popleft()                 # các nhóm bắt đầu cách đây k giá trị đã đủ k lá
    return opened == 0


print(hand_sweep(*ex1), hand_sweep(*ex2))
assert hand_sweep(*ex1) is True and hand_sweep(*ex2) is False

True False


In [11]:
# Code nộp LeetCode cho Cách 4
from collections import deque


class SolutionSweep:
    def isNStraightHand(self, hand: List[int], groupSize: int) -> bool:
        if len(hand) % groupSize:
            return False
        cnt = Counter(hand)
        q = deque()
        last, opened = -1, 0
        for x in sorted(cnt):
            if (opened > 0 and x != last + 1) or cnt[x] < opened:
                return False
            q.append(cnt[x] - opened)
            last, opened = x, cnt[x]
            if len(q) == groupSize:
                opened -= q.popleft()
        return opened == 0

**Độ phức tạp:** một lượt qua các giá trị khác nhau: `O(m log m)` (sắp xếp) + `O(m)`; **không** có vòng `for y in range(x, x + k)` nên không phụ thuộc `k`. Bộ nhớ `O(m)`.

## 8. TRACE từng bước

**Trace Cách 2** trên ví dụ 1: `hand = [1,2,3,6,2,3,4,7,8]`, `k = 3`. `cnt = {1:1, 2:2, 3:2, 4:1, 6:1, 7:1, 8:1}`; khoá sắp: `1, 2, 3, 4, 6, 7, 8`.

| `x` | `c = cnt[x]` | hành động | `cnt` sau |
|---|---|---|---|
| 1 | 1 | mở 1 nhóm `[1,2,3]`: trừ 1 ở `1, 2, 3` | `{1:0, 2:1, 3:1, 4:1, 6:1, 7:1, 8:1}` |
| 2 | 1 | mở 1 nhóm `[2,3,4]`: trừ 1 ở `2, 3, 4` | `{2:0, 3:0, 4:0, 6:1, 7:1, 8:1}` |
| 3 | 0 | bỏ qua | không đổi |
| 4 | 0 | bỏ qua | không đổi |
| 6 | 1 | mở nhóm `[6,7,8]` | tất cả `0` |
| 7, 8 | 0 | bỏ qua | — |

Duyệt xong → `True`. Hàm trace in từng bước (kèm một ví dụ `False`):

In [12]:
def trace_counter(hand, k):
    print(f"hand = {hand}, k = {k}")
    if len(hand) % k:
        print(f"{len(hand)} không chia hết cho {k} -> False")
        return False
    cnt = Counter(hand)
    show = lambda: {key: v for key, v in sorted(cnt.items()) if v}
    print("tần suất ban đầu:", show())
    for x in sorted(cnt):
        c = cnt[x]
        if not c:
            print(f"x={x}: đã hết lá -> bỏ qua")
            continue
        print(f"x={x}: còn {c} lá -> mở {c} nhóm bắt đầu từ {x}: {list(range(x, x + k))}")
        for y in range(x, x + k):
            if cnt[y] < c:
                print(f"   cần {c} lá {y} nhưng chỉ còn {cnt[y]} -> False")
                return False
            cnt[y] -= c
        print("   còn lại:", show())
    print("duyệt xong -> True")
    return True


assert trace_counter(*ex1) is True
print()
assert trace_counter([1, 1, 2, 2, 3, 4], 3) is False

hand = [1, 2, 3, 6, 2, 3, 4, 7, 8], k = 3
tần suất ban đầu: {1: 1, 2: 2, 3: 2, 4: 1, 6: 1, 7: 1, 8: 1}
x=1: còn 1 lá -> mở 1 nhóm bắt đầu từ 1: [1, 2, 3]
   còn lại: {2: 1, 3: 1, 4: 1, 6: 1, 7: 1, 8: 1}
x=2: còn 1 lá -> mở 1 nhóm bắt đầu từ 2: [2, 3, 4]
   còn lại: {6: 1, 7: 1, 8: 1}
x=3: đã hết lá -> bỏ qua
x=4: đã hết lá -> bỏ qua
x=6: còn 1 lá -> mở 1 nhóm bắt đầu từ 6: [6, 7, 8]
   còn lại: {}
x=7: đã hết lá -> bỏ qua
x=8: đã hết lá -> bỏ qua
duyệt xong -> True

hand = [1, 1, 2, 2, 3, 4], k = 3
tần suất ban đầu: {1: 2, 2: 2, 3: 1, 4: 1}
x=1: còn 2 lá -> mở 2 nhóm bắt đầu từ 1: [1, 2, 3]
   cần 2 lá 3 nhưng chỉ còn 1 -> False


**Trace Cách 4 (quét)** trên ví dụ 1: các giá trị `1,2,3,4,6,7,8` có số lá `1,2,2,1,1,1,1`; `k = 3`.

| `x` | `cnt[x]` | `opened` trước | kiểm tra | `q` sau khi thêm | `opened` sau | khi `len(q) == 3` |
|---|---|---|---|---|---|---|
| 1 | 1 | 0 | ok | `[1]` | 1 | — |
| 2 | 2 | 1 | `2 ≥ 1` ok | `[1, 1]` | 2 | — |
| 3 | 2 | 2 | `2 ≥ 2` ok | `[1, 1, 0]` | 2 | `len = 3`: `opened -= q.popleft() (1)` → 1; `q = [1, 0]` |
| 4 | 1 | 1 | ok | `[1, 0, 0]` | 1 | `len = 3`: `opened -= 1` → 0; `q = [0, 0]` |
| 6 | 1 | 0 | `opened = 0` nên không cần liên tiếp | `[0, 0, 1]` | 1 | `len = 3`: `opened -= 0` → 1; `q = [0, 1]` |
| 7 | 1 | 1 | `7 = 6 + 1` ok | `[0, 1, 0]` | 1 | `opened -= 0` → 1; `q = [1, 0]` |
| 8 | 1 | 1 | ok | `[1, 0, 0]` | 1 | `opened -= 1` → 0 |

Cuối: `opened == 0` → `True`. Hàm trace:

In [13]:
def trace_sweep(hand, k):
    if len(hand) % k:
        print("n % k != 0 -> False")
        return False
    cnt = Counter(hand)
    q = deque()
    last, opened = -1, 0
    print(f"{'x':>3} {'cnt':>4} {'opened':>7}  hành động")
    for x in sorted(cnt):
        if opened > 0 and x != last + 1:
            print(f"{x:>3} {cnt[x]:>4} {opened:>7}  đứt quãng (trước là {last}) -> False")
            return False
        if cnt[x] < opened:
            print(f"{x:>3} {cnt[x]:>4} {opened:>7}  không đủ lá cho {opened} nhóm đang mở -> False")
            return False
        new = cnt[x] - opened
        q.append(new)
        before = opened
        last, opened = x, cnt[x]
        act = f"mở thêm {new} nhóm mới (q={list(q)})"
        if len(q) == k:
            done = q.popleft()
            opened -= done
            act += f"; {done} nhóm đủ {k} lá -> opened = {opened}"
        print(f"{x:>3} {cnt[x]:>4} {before:>7}  {act}")
    print("cuối: opened =", opened, "->", opened == 0)
    return opened == 0


assert trace_sweep(*ex1) is True
print()
assert trace_sweep([1, 2, 3, 3, 4, 5, 7], 7) is False
print()
assert trace_sweep([1, 2, 4, 6], 2) is False

  x  cnt  opened  hành động
  1    1       0  mở thêm 1 nhóm mới (q=[1])
  2    2       1  mở thêm 1 nhóm mới (q=[1, 1])
  3    2       2  mở thêm 0 nhóm mới (q=[1, 1, 0]); 1 nhóm đủ 3 lá -> opened = 1
  4    1       1  mở thêm 0 nhóm mới (q=[1, 0, 0]); 1 nhóm đủ 3 lá -> opened = 0
  6    1       0  mở thêm 1 nhóm mới (q=[0, 0, 1]); 0 nhóm đủ 3 lá -> opened = 1
  7    1       1  mở thêm 0 nhóm mới (q=[0, 1, 0]); 0 nhóm đủ 3 lá -> opened = 1
  8    1       1  mở thêm 0 nhóm mới (q=[1, 0, 0]); 1 nhóm đủ 3 lá -> opened = 0
cuối: opened = 0 -> True

  x  cnt  opened  hành động
  1    1       0  mở thêm 1 nhóm mới (q=[1])
  2    1       1  mở thêm 0 nhóm mới (q=[1, 0])
  3    2       1  mở thêm 1 nhóm mới (q=[1, 0, 1])
  4    1       2  không đủ lá cho 2 nhóm đang mở -> False

  x  cnt  opened  hành động
  1    1       0  mở thêm 1 nhóm mới (q=[1])
  2    1       1  mở thêm 0 nhóm mới (q=[1, 0]); 1 nhóm đủ 2 lá -> opened = 0
  4    1       0  mở thêm 1 nhóm mới (q=[0, 1]); 0 nhóm đủ 2 lá ->

## 9. Code nộp LeetCode

Chọn **`Counter` + duyệt khoá đã sắp** (rõ ràng, `O(n + m log m)`). Giữ nguyên `class Solution`, `isNStraightHand(self, hand, groupSize)`:

In [14]:
from collections import Counter
from typing import List


class Solution:
    def isNStraightHand(self, hand: List[int], groupSize: int) -> bool:
        if len(hand) % groupSize:                   # số lá phải chia hết cho kích thước nhóm
            return False
        cnt = Counter(hand)
        for x in sorted(cnt):                       # từ lá nhỏ nhất: nó bắt buộc mở đầu một nhóm
            c = cnt[x]
            if c:
                for y in range(x, x + groupSize):   # mở c nhóm [x, x+1, ..., x+groupSize-1]
                    if cnt[y] < c:
                        return False
                    cnt[y] -= c
        return True


print(Solution().isNStraightHand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3), Solution().isNStraightHand([1, 2, 3, 4, 5], 4))
assert Solution().isNStraightHand(*ex1) is True and Solution().isNStraightHand(*ex2) is False

True False


## 10. Vì sao đúng

**Định lý (tính đúng của tham lam).** Nếu bộ bài chia được thành các nhóm `k` lá liên tiếp thì lá nhỏ nhất `m` thuộc một nhóm, và nhóm đó **bắt buộc** là `{m, m+1, …, m+k−1}`; hơn nữa bộ bài còn lại sau khi bỏ nhóm này vẫn chia được.

*Chứng minh.* Gọi `G` là nhóm chứa một lá `m` (nhỏ nhất toàn bộ). Các lá của `G` là `k` giá trị liên tiếp `a, a+1, …, a+k−1` với `a ≤ m` (vì `m ∈ G`). Mà `m` nhỏ nhất toàn bộ nên mọi lá `≥ m`, do đó `a ≥ m`; suy ra `a = m`. Vậy `G = {m, …, m+k−1}` (đúng một lá mỗi giá trị). Bỏ `G` ra thì các nhóm còn lại của cách chia ban đầu vẫn là cách chia của phần còn lại. ∎

**Cách 2 (theo số lượng).** Xét giá trị nhỏ nhất `x` còn bài với `c = cnt[x]` lá. Mỗi lá `x` là lá nhỏ nhất (còn lại) của một nhóm — ta áp dụng định lý `c` lần liên tiếp — nên cần `c` nhóm `{x, …, x+k−1}` **rời nhau** (về lá bài), tức mỗi `y ∈ [x, x+k−1]` cần ít nhất `c` lá. Nếu thiếu → không chia được. Ngược lại trừ `c` ở mỗi `y`: giá trị `x` về `0`, và ta tiếp tục với giá trị nhỏ nhất kế tiếp. Nếu mọi khoá đều qua được thì mọi lá đã được xếp vào nhóm.

**Điều kiện `n % k`.** Mỗi nhóm có đúng `k` lá nên tổng lá chia hết cho `k`; vòng lặp tham lam cũng sẽ thất bại nếu không chia hết, nên kiểm tra sớm chỉ để thoát nhanh.

**Cách 3 (heap).** `heap[0]` luôn là giá trị nhỏ nhất còn lá (ta chỉ `heappop` giá trị khi nó hết lá và đang là đỉnh); mỗi vòng mở **một** nhóm bắt đầu từ đó, tương đương áp dụng định lý một lần. Điều kiện `y != heap[0]` khi `cnt[y]` về `0` bảo đảm không để lại giá trị nhỏ hơn `y` còn lá trong khi `y` đã hết (nếu có, giá trị nhỏ hơn đó sẽ phải bắt đầu một nhóm không chứa `y`... thực ra nó chắc chắn sẽ thất bại sau; ta phát hiện sớm).

**Cách 4 (quét).** Ý nghĩa của `opened` tại giá trị `x`: số nhóm **đã bắt đầu** ở một trong `k − 1` giá trị trước và chưa đủ `k` lá. Các nhóm đó mỗi nhóm cần một lá `x` (do liên tiếp) nên cần `cnt[x] ≥ opened`; các lá `x` còn thừa mở nhóm mới. Nhóm bắt đầu tại giá trị `x₀` hoàn thành khi nhận lá `x₀ + k − 1`; hàng đợi `q` nhớ số nhóm bắt đầu ở mỗi giá trị trong cửa sổ `k` giá trị liền nhau (vì giá trị liên tiếp, độ dài `q` bằng số giá trị đã đi qua kể từ khi bắt đầu; chỉ khi `opened > 0` nó liên tiếp — ta không cần `q` phản ánh các khoảng trống vì khi `opened = 0` hàng đợi chứa toàn `0`). Cuối cùng `opened == 0` nghĩa là không nhóm nào dở dang.

Ta kiểm chứng bằng máy: các cách khớp **oracle liệt kê mọi cách chia nhóm** (không giả sử tham lam) trên **mọi** bộ bài nhỏ (mục 12D).

## 11. Time / Space Complexity

Đặt `n = len(hand)`, `m` = số giá trị khác nhau, `k = groupSize`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Sắp xếp + `list.remove` | `O(n²)` | `O(1)` thêm | `10^8` ở `n = 10^4`: chậm |
| 2. `Counter` + khoá đã sắp | `O(n + m log m)` | `O(m)` | **đáp án chính** |
| 3. Min-heap | `O(n + m log m)` | `O(m)` | mỗi vòng một nhóm |
| 4. Quét + nhóm đang mở | `O(m log m)` | `O(m)` | không phụ thuộc `k` |

Đo thực tế (`n = 10^4`, các `k`):

In [15]:
def planted_hand(rng, n, k, spread=10 ** 9):
    """Bộ bài chia được: n/k nhóm liên tiếp với điểm bắt đầu ngẫu nhiên, rồi xáo trộn."""
    hand = []
    for _ in range(n // k):
        start = rng.randint(0, spread - k)
        hand += list(range(start, start + k))
    rng.shuffle(hand)
    return hand


def bench(fn, hand, k, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(hand, k)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(846)
for n, k, spread in [(10 ** 4, 1, 10 ** 9), (10 ** 4, 10, 10 ** 9), (10 ** 4, 100, 150), (10 ** 4, 1000, 1500), (10 ** 4, 10 ** 4, 10 ** 4)]:
    hand = planted_hand(rng, n, k, spread)
    ref = hand_counter(hand, k)
    row = []
    for name, fn in [("counter", hand_counter), ("heap", hand_heap), ("sweep", hand_sweep)]:
        assert fn(hand, k) is ref
        row.append(f"{name} {bench(fn, hand, k) * 1000:7.2f} ms")
    if n * 1 <= 10 ** 4 and k >= 10:
        row.append(f"remove {bench(hand_remove, hand, k, 1) * 1000:8.1f} ms")
    print(f"n={n} k={k:5d}: đáp án {ref!s:5} | " + " | ".join(row))

n=10000 k=    1: đáp án True  | counter    5.01 ms | heap    7.43 ms | sweep    3.58 ms


n=10000 k=   10: đáp án True  | counter    4.18 ms | heap    5.34 ms | sweep    3.70 ms | remove      8.7 ms


n=10000 k=  100: đáp án True  | counter    0.69 ms | heap    1.63 ms | sweep    0.22 ms | remove    116.7 ms


n=10000 k= 1000: đáp án True  | counter    2.89 ms | heap    2.94 ms | sweep    0.88 ms | remove    120.9 ms


n=10000 k=10000: đáp án True  | counter    4.61 ms | heap    6.09 ms | sweep    3.83 ms | remove      9.2 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **vét cạn mọi bộ bài nhỏ** với oracle **liệt kê mọi cách chia nhóm** (không giả sử tham lam), (E) ngẫu nhiên có seed (gieo nhóm hợp lệ / làm hỏng), (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [16]:
def oracle(hand, k):
    """Oracle độc lập: liệt kê MỌI cách chia hand thành nhóm k lá (không giả sử tham lam) và kiểm tra nhóm liên tiếp."""
    n = len(hand)
    if n % k:
        return False
    cards = list(hand)

    def solve(remaining):
        if not remaining:
            return True
        first, rest = remaining[0], remaining[1:]
        for others in combinations(range(len(rest)), k - 1):
            group = sorted([first] + [rest[i] for i in others])
            if all(group[i + 1] - group[i] == 1 for i in range(k - 1)):
                left = [rest[i] for i in range(len(rest)) if i not in others]
                if solve(left):
                    return True
        return False

    return solve(cards)


IMPLS = {
    "remove": hand_remove,
    "counter": hand_counter,
    "heap": hand_heap,
    "sweep": hand_sweep,
    "Solution": lambda h, k: Solution().isNStraightHand(list(h), k),
    "SolutionCounter": lambda h, k: SolutionCounter().isNStraightHand(list(h), k),
    "SolutionHeap": lambda h, k: SolutionHeap().isNStraightHand(list(h), k),
    "SolutionSweep": lambda h, k: SolutionSweep().isNStraightHand(list(h), k),
}


def check(hand, k, want=None, names=None):
    if want is None:
        want = oracle(hand, k)
    for name in (names or IMPLS):
        got = IMPLS[name](list(hand), k)
        assert got is want, (name, hand, k, got, want)


check(*ex1, True)
check(*ex2, False)
assert oracle(*ex1) is True and oracle(*ex2) is False
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt và oracle liệt kê mọi cách chia đều đúng")

ví dụ 1, 2: cả 8 cài đặt và oracle liệt kê mọi cách chia đều đúng


### B. Edge case

- `groupSize = 1`: luôn `True`.
- `groupSize = n`: `True` khi và chỉ khi các giá trị là `n` số liên tiếp (không trùng).
- `n % groupSize != 0` → `False`.
- Các lá **trùng** nhiều lần.
- Có **khoảng hở** giữa các giá trị (không liên tiếp).
- Giá trị bằng `0` và giá trị rất lớn `10^9`.
- Hai nhóm chồng lên nhau (giá trị chung): `[1,2,2,3,3,4]`, `k = 3`.
- Một giá trị lặp nhiều hơn các giá trị khác (`[1,1,1,2,3]`, `k = 3`).

In [17]:
cases = [
    ([5], 1, True), ([3, 1, 2, 9], 1, True),
    ([1, 2, 3, 4, 5], 5, True), ([1, 2, 3, 4, 6], 5, False),
    ([1, 2, 3, 4, 5], 2, False),                       # 5 % 2 != 0
    ([1, 1, 2, 2, 3, 3], 3, True), ([1, 1, 2, 2, 3, 4], 3, False),
    ([1, 2, 2, 3, 3, 4], 3, True), ([1, 1, 1, 2, 3], 3, False),
    ([0, 1, 2, 10 ** 9 - 2, 10 ** 9 - 1, 10 ** 9], 3, True),
    ([1, 2, 4, 5], 2, True), ([1, 2, 4, 6], 2, False), ([1, 3, 5, 7], 2, False),
    ([8, 10, 12], 3, False), ([1, 2, 3, 6, 2, 3, 4, 7, 8], 3, True), ([1, 2, 3, 6, 2, 3, 4, 7, 9], 3, False),
    ([7, 7, 7, 7], 4, False), ([7, 7, 8, 8], 2, True),
]
for hand, k, want in cases:
    check(hand, k, want)
    assert oracle(hand, k) is want, (hand, k)
print(len(cases), "edge case: tất cả cài đặt và oracle khớp đáp án viết tay")

18 edge case: tất cả cài đặt và oracle khớp đáp án viết tay


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_set_presence(hand, k):                    # chỉ kiểm tra có mặt, bỏ qua số lượng
    if len(hand) % k:
        return False
    s = set(hand)
    return all(all((x + i) in s for i in range(k)) for x in s if (x - 1) not in s) or all(
        any(all((start + i) in s for i in range(k)) for start in range(x - k + 1, x + 1)) for x in s)


def bug_no_divisibility(hand, k):                 # quên kiểm tra n % k và trừ chỉ 1 mỗi lần, thoả khi mỗi giá trị có nhóm
    cnt = Counter(hand)
    for x in sorted(cnt):
        if cnt[x] > 0:
            for y in range(x, x + k):
                if cnt[y] <= 0:
                    return False
                cnt[y] -= 1
    return all(v <= 0 for v in cnt.values())      # SAI về logic: một lượt chỉ mở một nhóm cho mỗi x


def bug_start_any(hand, k):                       # bắt đầu nhóm từ lá đầu tiên theo thứ tự xuất hiện (không sắp xếp)
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    for x in hand:                                # SAI: không duyệt theo thứ tự tăng dần
        if cnt[x] > 0:
            for y in range(x, x + k):
                if cnt[y] <= 0:
                    return False
                cnt[y] -= 1
    return True


def bug_decrement_by_one(hand, k):                # sửa: duyệt tăng dần nhưng chỉ trừ 1 dù c > 1
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    for x in sorted(cnt):
        if cnt[x] > 0:
            for y in range(x, x + k):
                if cnt[y] <= 0:
                    return False
                cnt[y] -= 1                       # SAI: cần trừ c, và lặp lại tới khi cnt[x] == 0
    return True


def bug_heap_pop_always(hand, k):                 # min-heap nhưng pop giá trị đầu nhóm sau MỖI nhóm dù còn lá
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    heap = list(cnt)
    heapify(heap)
    while heap:
        x = heap[0]
        for y in range(x, x + k):
            if cnt[y] == 0:
                return False
            cnt[y] -= 1
        heappop(heap)                             # SAI: x có thể còn lá (cnt[x] > 0)
    return True


BUGS = [bug_set_presence, bug_no_divisibility, bug_start_any, bug_decrement_by_one, bug_heap_pop_always]
rng = random.Random(1)
tests = []
for _ in range(1500):
    k = rng.randint(1, 4)
    n = k * rng.randint(1, 3)
    tests.append(([rng.randint(0, 4) for _ in range(n)], k))
tests += [([1, 1, 2, 2, 3, 3], 3), ([1, 2, 2, 3, 3, 4], 3), ([1, 1, 2, 2, 3, 4], 3)]
for bug in BUGS:
    bad = None
    for h, k in tests:
        try:
            got = bug(list(h), k)
        except Exception as e:
            got = type(e).__name__
        if got != oracle(h, k):
            bad = (h, k, got)
            break
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên hand={bad[0]}, k={bad[1]}: ra {bad[2]}, đúng {oracle(bad[0], bad[1])}")

bug_set_presence       sai trên hand=[0, 3, 2, 1, 4, 0, 2, 0, 0, 0, 4, 0], k=4: ra True, đúng False
bug_no_divisibility    sai trên hand=[0, 0, 0], k=1: ra False, đúng True
bug_start_any          sai trên hand=[2, 0, 1, 1], k=2: ra False, đúng True
bug_decrement_by_one   sai trên hand=[3, 1, 4, 4, 0, 3], k=2: ra False, đúng True
bug_heap_pop_always    sai trên hand=[2, 0, 1, 1], k=2: ra False, đúng True


### D. Vét cạn mọi bộ bài nhỏ

Mọi **đa tập** (multiset) các lá bài gồm `n ≤ 8` lá với giá trị trong `{0, 1, 2, 3, 4}` (nhiều lá trùng) và mọi `groupSize` là ước của `n` (kể cả `groupSize = 1`) — khoảng vài nghìn ca. Oracle `oracle` **liệt kê mọi cách chia nhóm**, không giả sử tham lam.

In [19]:
t0 = time.perf_counter()
cnt_cases = cnt_true = 0
for n in range(1, 9):
    for hand in combinations_with_replacement(range(5), n):
        for k in range(1, n + 1):
            if n % k:
                continue
            want = oracle(list(hand), k)
            for name in IMPLS:
                assert IMPLS[name](list(hand), k) is want, (name, hand, k)
            cnt_cases += 1
            cnt_true += want
print(cnt_cases, "(bộ bài, groupSize): 8 cài đặt khớp oracle; số ca True =", cnt_true, f"; {time.perf_counter() - t0:.1f} s")
# ngoài ước: n % k != 0 -> False với mọi cài đặt
for hand in combinations_with_replacement(range(4), 5):
    for k in (2, 3, 4):
        for name in IMPLS:
            assert IMPLS[name](list(hand), k) is False
print("các ca n % k != 0: mọi cài đặt trả False")

4047 (bộ bài, groupSize): 8 cài đặt khớp oracle; số ca True = 1370 ; 0.1 s
các ca n % k != 0: mọi cài đặt trả False


### E. Ngẫu nhiên có seed

Bộ bài **gieo** từ các nhóm liên tiếp (chắc chắn `True`), rồi **làm hỏng** bằng cách đổi một lá, thay đổi một giá trị, hoặc thêm/bớt lá; và các bộ bài ngẫu nhiên thuần. Oracle thay bằng cách nhanh độc lập (`oracle`) khi `n ≤ 9`, còn lớn hơn thì so sánh các cách nhanh và kiểm tra tính chất.

In [20]:
rng = random.Random(20240846)
cnt_plant = cnt_corrupt = cnt_small = 0
for _ in range(1500):
    k = rng.randint(1, 6)
    groups = rng.randint(1, 8)
    spread = rng.choice([k + 3, 2 * k + 5, 10 ** 9])
    hand = planted_hand(rng, groups * k, k, spread)
    check(hand, k, True, [x for x in IMPLS if x != "remove"] if len(hand) > 30 else None)
    cnt_plant += 1
    t = list(hand)
    mode = rng.random()
    i = rng.randrange(len(t))
    if mode < 0.4:
        t[i] += rng.choice([-1, 1, 2])
    elif mode < 0.7:
        t.pop(i)
        t.append(rng.randint(0, spread))
    else:
        t[i] = rng.randint(0, spread)
    want = Solution().isNStraightHand(list(t), k)
    for name in IMPLS:
        assert IMPLS[name](list(t), k) is want, (name, t, k)
    if len(t) <= 9:
        assert want is oracle(t, k)
        cnt_small += 1
    cnt_corrupt += 1
print(f"gieo {cnt_plant} bộ bài hợp lệ: mọi cài đặt True; làm hỏng {cnt_corrupt} ({cnt_small} ca nhỏ đối chiếu oracle): các cài đặt khớp nhau")

# tính chất: hoán vị bài không đổi kết quả; thêm một nhóm liên tiếp vào bộ bài chia được thì vẫn chia được
for _ in range(500):
    k = rng.randint(1, 5)
    hand = [rng.randint(0, 6) for _ in range(k * rng.randint(1, 4))]
    base = Solution().isNStraightHand(list(hand), k)
    sh = list(hand); rng.shuffle(sh)
    assert Solution().isNStraightHand(sh, k) is base
    start = rng.randint(0, 10)
    if base:                                              # bộ bài chia được + thêm một nhóm liên tiếp vẫn chia được
        assert Solution().isNStraightHand(hand + list(range(start, start + k)), k) is True
print("tính chất: hoán vị không đổi; thêm một nhóm liên tiếp vào bộ bài chia được vẫn chia được — OK")

gieo 1500 bộ bài hợp lệ: mọi cài đặt True; làm hỏng 1500 (608 ca nhỏ đối chiếu oracle): các cài đặt khớp nhau
tính chất: hoán vị không đổi; thêm một nhóm liên tiếp vào bộ bài chia được vẫn chia được — OK


### F. Giới hạn lớn nhất

`n = 10^4`: `groupSize = 1`, `groupSize = n`, nhiều nhóm nhỏ, giá trị sát `10^9`, một nhóm duy nhất khổng lồ, các nhóm chồng giá trị, và một ca `False` khó (gần đúng).

In [21]:
n = 10 ** 4
rng = random.Random(777)
fast = [x for x in IMPLS if x != "remove"]
big_cases = {
    "k = 1": (planted_hand(rng, n, 1), 1, True),
    "k = n (n số liên tiếp)": (list(range(10 ** 9 - n + 1, 10 ** 9 + 1)), n, True),
    "100 nhóm 100 lá": (planted_hand(rng, n, 100, 10 ** 9), 100, True),
    "nhóm chồng giá trị (spread nhỏ)": (planted_hand(rng, n, 50, 120), 50, True),
    "10^4 lá, k = 2, spread 5": (planted_hand(rng, n, 2, 7), 2, True),
}
bad = planted_hand(rng, n, 100, 10 ** 9)
bad[0] += 1                                           # làm hỏng đúng một lá
big_cases["gần đúng (một lá lệch)"] = (bad, 100, None)
for name, (hand, k, want) in big_cases.items():
    ref = hand_counter(hand, k)
    if want is not None:
        assert ref is want, (name, ref)
    for impl in fast:
        assert IMPLS[impl](hand, k) is ref, (name, impl)
    print(f"{name:34s}: {ref} (7 cài đặt khớp)")
assert hand_counter(*big_cases["gần đúng (một lá lệch)"][:2]) is False

k = 1                             : True (7 cài đặt khớp)
k = n (n số liên tiếp)            : True (7 cài đặt khớp)
100 nhóm 100 lá                   : True (7 cài đặt khớp)
nhóm chồng giá trị (spread nhỏ)   : True (7 cài đặt khớp)
10^4 lá, k = 2, spread 5          : True (7 cài đặt khớp)
gần đúng (một lá lệch)            : False (7 cài đặt khớp)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ kiểm tra "có mặt" (dùng `set`), quên số lượng

Bộ bài `[1,1,2,2,3,3]` chia được thành `[1,2,3]` hai lần. Nhưng `[1,1,2,3,3,4]`? Các giá trị `1,2,3,4` đều có mặt, mà cần `2` lá `1` cho hai nhóm nên cần `2` lá `2`; chỉ có `1` lá `2`. Số lượng quyết định.

In [22]:
h, k = [1, 1, 2, 3, 3, 4], 3
print("bản dùng set:", bug_set_presence(list(h), k), "| đúng:", Solution().isNStraightHand(list(h), k), "| oracle:", oracle(h, k))
bad = next((t for t in tests if bug_set_presence(list(t[0]), t[1]) != oracle(*t)))
print("ví dụ khác:", bad, "->", bug_set_presence(list(bad[0]), bad[1]), "đúng", oracle(*bad))
assert bug_set_presence(list(h), k) != oracle(h, k)

bản dùng set: True | đúng: False | oracle: False
ví dụ khác: ([0, 3, 2, 1, 4, 0, 2, 0, 0, 0, 4, 0], 4) -> True đúng False


**Vì sao sai:** `set` không phân biệt một lá và nhiều lá cùng giá trị. **Đúng:** `Counter` và so sánh `cnt[y] >= c`.

### Lỗi 2: bắt đầu nhóm từ lá không nhỏ nhất (không sắp xếp)

Nếu duyệt `hand` theo thứ tự xuất hiện và bắt đầu nhóm ở lá bất kỳ, ta có thể dùng nhầm lá làm đầu nhóm trong khi còn lá nhỏ hơn.

In [23]:
h, k = [2, 3, 4, 1, 2, 3], 3                       # đúng: [1,2,3] và [2,3,4]
print("bắt đầu theo thứ tự xuất hiện:", bug_start_any(list(h), k), "| đúng:", Solution().isNStraightHand(list(h), k))
bad = next(((t, kk) for t, kk in tests if bug_start_any(list(t), kk) != oracle(t, kk)))
print("ví dụ khác:", bad, "->", bug_start_any(list(bad[0]), bad[1]), "đúng", oracle(*bad))
assert bug_start_any(list(h), k) is False and Solution().isNStraightHand(list(h), k) is True

bắt đầu theo thứ tự xuất hiện: False | đúng: True
ví dụ khác: ([2, 0, 1, 1], 2) -> False đúng True


**Vì sao sai:** từ lá `2` đầu tiên, nhóm `[2,3,4]` bị tạo trước, nhưng lá `1` còn lại không thể làm đầu nhóm nữa (thiếu bài). Lá nhỏ nhất phải được xử lý trước. **Đúng:** `for x in sorted(cnt)`.

### Lỗi 3: trừ `1` thay vì trừ `c` (số lá của lá đầu nhóm)

Khi `cnt[x] = c > 1`, cần mở **`c` nhóm** cùng lúc. Chỉ trừ `1` rồi sang giá trị tiếp theo bỏ sót.

In [24]:
h, k = [1, 1, 2, 2, 3, 3], 3
print("chỉ trừ 1:", bug_decrement_by_one(list(h), k), "| đúng:", Solution().isNStraightHand(list(h), k))
bad = next(((t, kk) for t, kk in tests if bug_decrement_by_one(list(t), kk) != oracle(t, kk)))
print("ví dụ khác:", bad, "->", bug_decrement_by_one(list(bad[0]), bad[1]), "đúng", oracle(*bad))

chỉ trừ 1: False | đúng: True
ví dụ khác: ([3, 1, 4, 4, 0, 3], 2) -> False đúng True


**Vì sao sai:** với `[1,1,2,2,3,3]`, sau khi trừ `1` ở `x = 1` vẫn còn một lá `1` chưa nằm trong nhóm nào, nhưng vòng lặp chuyển sang `x = 2` và mở nhóm `[2,3,4]` từ lá `2` (cần lá `4` không có) nên trả `False` sai. **Đúng:** `cnt[y] -= c` cho mọi `y` trong nhóm (hoặc lặp tới khi `cnt[x] == 0`).

### Lỗi 4: min-heap — `heappop` giá trị đầu nhóm sau mỗi nhóm dù còn lá

Mỗi giá trị khác nhau chỉ nằm **một lần** trong heap; chỉ được `heappop` khi **hết lá** của giá trị đó. Nếu pop ngay sau khi lấy một nhóm (dù `cnt[x] > 0`), các lá `x` còn lại không bao giờ được xét làm đầu nhóm nữa.

In [25]:
h, k = [1, 1, 2, 2, 3, 3], 3
print("pop luôn:", bug_heap_pop_always(list(h), k), "| đúng:", Solution().isNStraightHand(list(h), k))
bad = next(((t, kk) for t, kk in tests if bug_heap_pop_always(list(t), kk) != oracle(t, kk)), None)
print("một ca sai khác:", bad, "->", bug_heap_pop_always(list(bad[0]), bad[1]), "| đúng:", oracle(*bad))
assert bad is not None

pop luôn: False | đúng: True
một ca sai khác: ([2, 0, 1, 1], 2) -> False | đúng: True


**Vì sao sai:** heap chứa các **giá trị**, không phải từng lá; mỗi nhóm chỉ trừ một lá. **Đúng:** `cnt[y] -= 1; if cnt[y] == 0: heappop` (và kiểm tra `y == heap[0]`), hoặc trừ `c` một lần như Cách 2.

### Lỗi 5: dùng mảng đếm theo giá trị

Giá trị tới `10^9`: mảng đếm `10^9` phần tử không thể. Dùng `Counter`/từ điển.

In [26]:
print("mảng đếm theo giá trị 10^9 phần tử cần ~", 10 ** 9 * 8 / 1e9, "GB (mỗi phần tử 8 byte)")
print("Counter cho 10^4 lá bài giá trị tới 10^9 chỉ có tối đa 10^4 khoá")

mảng đếm theo giá trị 10^9 phần tử cần ~ 8.0 GB (mỗi phần tử 8 byte)
Counter cho 10^4 lá bài giá trị tới 10^9 chỉ có tối đa 10^4 khoá


**Vì sao sai:** bộ nhớ. **Đúng:** từ điển theo giá trị xuất hiện.

### Lỗi 6 (bonus): sửa `Counter` trong khi duyệt chính nó

`for x in cnt: ... del cnt[x]` gây `RuntimeError: dictionary changed size during iteration`. Duyệt trên `sorted(cnt)` (một **danh sách** sao chép khoá) thì an toàn.

In [27]:
c = Counter([1, 2, 3])
try:
    for x in c:
        c[x + 10] = 1
except RuntimeError as e:
    print("RuntimeError:", e)
c = Counter([1, 2, 3])
for x in sorted(c):
    c[x + 10] = 1
print("duyệt trên sorted(c):", dict(sorted(c.items())))

RuntimeError: dictionary changed size during iteration
duyệt trên sorted(c): {1: 1, 2: 1, 3: 1, 11: 1, 12: 1, 13: 1}


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chia tập các số thành các nhóm có dạng đã định (liên tiếp, kích thước cố định)" → **tham lam từ phần tử nhỏ nhất** + **đếm tần suất**.

**Mẹo:**

1. Lá nhỏ nhất **buộc** mở đầu một nhóm: đây là lập luận tham lam chuẩn (exchange argument ngắn).
2. Dùng `Counter` và duyệt khoá đã sắp; khi gặp khoá có `c` lá, mở **`c`** nhóm cùng lúc (trừ `c`).
3. Kiểm tra `len % k` để thoát sớm.
4. Giá trị lớn → từ điển, không mảng.
5. Biến thể O(n log n) không phụ thuộc `k`: **nhóm đang mở** + hàng đợi.
6. Oracle: liệt kê mọi cách chia (nhóm chứa lá đầu tiên) cho bộ bài nhỏ.

**Họ bài cùng pattern:** Divide Array in Sets of K Consecutive Numbers (1296, cùng đề), Split Array into Consecutive Subsequences (659, nhóm độ dài `≥ 3`), Rearrange Array to Make Group (954), Maximum Number of Groups Entering a Competition (2358), Longest Consecutive Sequence (128).

## 15. Biến thể

1. **Trả về chính các nhóm** (bài tập 5).
2. **Độ dài nhóm `≥ k`** (Split Array into Consecutive Subsequences, 659): khác hẳn, nhóm có thể dài hơn.
3. **Số nhóm tối đa** khi không bắt buộc dùng hết bài.
4. **Số lá bài cần thêm** ít nhất để chia được.
5. **Nhóm gồm các giá trị cách nhau `d`** (cấp số cộng): đổi `range(x, x + k)` thành `range(x, x + k·d, d)`.

Cài đặt biến thể 5 (cấp số cộng, bước `d`) và so sánh với oracle liệt kê cách chia:

In [28]:
def hand_step(hand, k, d=1):
    """Biến thể 5: mỗi nhóm là x, x+d, ..., x+(k-1)d."""
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    for x in sorted(cnt):
        c = cnt[x]
        if c:
            for i in range(k):
                y = x + i * d
                if cnt[y] < c:
                    return False
                cnt[y] -= c
    return True


def oracle_step(hand, k, d):
    n = len(hand)
    if n % k:
        return False

    def solve(rem):
        if not rem:
            return True
        first, rest = rem[0], rem[1:]
        for others in combinations(range(len(rest)), k - 1):
            g = sorted([first] + [rest[i] for i in others])
            if all(g[i + 1] - g[i] == d for i in range(k - 1)):
                if solve([rest[i] for i in range(len(rest)) if i not in others]):
                    return True
        return False

    return solve(list(hand))


assert hand_step(*ex1) is True and hand_step([1, 3, 5, 2, 4, 6], 3, 2) is True and hand_step([1, 2, 3], 3, 2) is False
rng = random.Random(15)
for _ in range(1500):
    k = rng.randint(1, 3)
    n = k * rng.randint(1, 3)
    d = rng.randint(1, 3)
    h = [rng.randint(0, 7) for _ in range(n)]
    assert hand_step(h, k, d) is oracle_step(h, k, d), (h, k, d)
print("biến thể bước d khớp oracle liệt kê cách chia trên 1500 ca")

biến thể bước d khớp oracle liệt kê cách chia trên 1500 ca


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp + `list.remove` | Mỗi nhóm bắt đầu từ lá nhỏ nhất, xoá `k` lá | `O(n²)` | `O(1)` | Oracle nhanh; `n` nhỏ |
| 2. `Counter` + khoá đã sắp | Mở `c` nhóm tại khoá `x` có `c` lá | `O(n + m log m)` | `O(m)` | **Đáp án chính** |
| 3. Min-heap | Luôn lấy giá trị nhỏ nhất còn lá, mở một nhóm | `O(n + m log m)` | `O(m)` | Khi bài cập nhật động |
| 4. Quét + nhóm đang mở | Chỉ nhớ số nhóm mở (hàng đợi `k` ô) | `O(m log m)` | `O(m)` | Không phụ thuộc `k` |

Cả bốn khớp oracle liệt kê mọi cách chia trên toàn bộ bộ bài nhỏ vét cạn và hàng nghìn bộ bài ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [29]:
for hand, k in ([1, 2, 3, 4], 2), ([1, 2, 3, 4], 3), ([1, 1, 2, 2], 2), ([3, 3, 4, 4, 5, 5], 3), ([1, 3, 5], 1):
    print(hand, k, "->", Solution().isNStraightHand(hand, k))

[1, 2, 3, 4] 2 -> True
[1, 2, 3, 4] 3 -> False
[1, 1, 2, 2] 2 -> True
[3, 3, 4, 4, 5, 5] 3 -> True
[1, 3, 5] 1 -> True


### Bài 2 (tính tay)

`hand = [5, 3, 4, 4, 6, 5, 7, 8]`, `groupSize = 4`.

- (a) Sắp xếp và đếm tần suất.
- (b) Chạy tham lam bằng tay: các nhóm nào được lấy?
- (c) Đổi `8` thành `9`: kết quả đổi thế nào và vì sao?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách `Counter`:

In [30]:
def counter_todo(hand, k):
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    for x in sorted(cnt):
        c = cnt[x]
        if c:
            pass            # TODO: với mỗi y trong [x, x+k): nếu cnt[y] < c trả False; ngược lại cnt[y] -= c
    return True


# Khi làm xong, bỏ comment để kiểm tra:
# assert counter_todo([1, 2, 3, 6, 2, 3, 4, 7, 8], 3) is True and counter_todo([1, 2, 3, 4, 5], 4) is False
# assert counter_todo([1, 1, 2, 2, 3, 4], 3) is False

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` theo ý tưởng min-heap hoặc `Counter`:

In [31]:
class MySolution:
    def isNStraightHand(self, hand, groupSize):
        pass            # TODO


# Khi làm xong:
# assert MySolution().isNStraightHand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3) is True
# assert MySolution().isNStraightHand([1, 2, 3, 4, 5], 4) is False

### Bài 5 (mini challenge): trả về chính các nhóm

Viết `build_groups(hand, groupSize)` trả về **danh sách các nhóm** (mỗi nhóm là list `groupSize` giá trị liên tiếp) nếu chia được, ngược lại `None`. Các nhóm phải dùng đúng mọi lá (đúng đa tập). Kiểm bằng hàm hợp lệ (nhiều cách chia có thể khác thứ tự).

In [32]:
def build_groups(hand, groupSize):
    pass            # TODO


# Khi làm xong (kiểm bằng hàm hợp lệ):
# g = build_groups([1, 2, 3, 6, 2, 3, 4, 7, 8], 3)
# assert g is not None and sorted(v for grp in g for v in grp) == sorted([1, 2, 3, 6, 2, 3, 4, 7, 8])

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `n % k` trước.
- Mức 2: `[1,2,3,4]`, `k = 2`: `[1,2]`, `[3,4]` ✓; `k = 3`: `4 % 3 ≠ 0`.
- Mức 3: kết quả `True, False, True, True, True`: `[1,1,2,2]` với `k = 2` chia được thành `[1,2]` hai lần; `[3,3,4,4,5,5]` với `k = 3` chia được thành `[3,4,5]` hai lần; `[1,3,5]` với `k = 1` luôn được.

**Bài 2**

- Mức 1: sắp: `3 4 4 5 5 6 7 8`; tần suất `{3:1, 4:2, 5:2, 6:1, 7:1, 8:1}`.
- Mức 2: lá nhỏ nhất `3` → `[3,4,5,6]`; còn `4 5 7 8`: lá nhỏ nhất `4` → cần `5, 6, 7` nhưng `6` đã hết.
- Mức 3: (b) không chia được → `False`; (c) với `9` thay `8`: cũng `False`. Hãy kiểm bằng `trace_counter`.

**Bài 3**

- Mức 1: `for y in range(x, x + k)`.
- Mức 2: `if cnt[y] < c: return False`.
- Mức 3: `cnt[y] -= c`.

**Bài 4**

- Mức 1: kiểm `n % k`.
- Mức 2: `Counter` + khoá đã sắp, hoặc heap các khoá.
- Mức 3: xem mục 9.

**Bài 5**

- Mức 1: như Cách 2 nhưng ghi nhóm.
- Mức 2: với `c` lá `x`, thêm `c` nhóm `[x..x+k-1]`.
- Mức 3: kiểm tra bằng đa tập.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [33]:
pairs = [([1, 2, 3, 4], 2), ([1, 2, 3, 4], 3), ([1, 1, 2, 2], 2), ([3, 3, 4, 4, 5, 5], 3), ([1, 3, 5], 1)]
res = [Solution().isNStraightHand(h, k) for h, k in pairs]
assert res == [True, False, True, True, True]
assert res == [oracle(h, k) for h, k in pairs]
print("Bài 1 OK:", res)

Bài 1 OK: [True, False, True, True, True]


### Bài 2

(a) Sắp xếp: `3 4 4 5 5 6 7 8`; tần suất `{3:1, 4:2, 5:2, 6:1, 7:1, 8:1}`. (b) Lá nhỏ nhất `3` → `[3,4,5,6]`; còn `4 5 7 8`: lá `4` cần `5, 6, 7` mà `6` đã hết → `False`. (c) Thay `8` bằng `9`: `3 4 4 5 5 6 7 9`: sau `[3,4,5,6]` còn `4 5 7 9`: `4` cần `5, 6, 7` thiếu `6` → vẫn `False`.

In [34]:
h = [5, 3, 4, 4, 6, 5, 7, 8]
assert sorted(h) == [3, 4, 4, 5, 5, 6, 7, 8]
assert dict(sorted(Counter(h).items())) == {3: 1, 4: 2, 5: 2, 6: 1, 7: 1, 8: 1}
assert Solution().isNStraightHand(h, 4) is False and oracle(h, 4) is False
h2 = [5, 3, 4, 4, 6, 5, 7, 9]
assert Solution().isNStraightHand(h2, 4) is False and oracle(h2, 4) is False
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def counter_answer(hand, k):
    if len(hand) % k:
        return False
    cnt = Counter(hand)
    for x in sorted(cnt):
        c = cnt[x]
        if c:
            for y in range(x, x + k):
                if cnt[y] < c:
                    return False
                cnt[y] -= c
    return True


assert counter_answer([1, 2, 3, 6, 2, 3, 4, 7, 8], 3) is True and counter_answer([1, 2, 3, 4, 5], 4) is False
assert counter_answer([1, 1, 2, 2, 3, 4], 3) is False
rng = random.Random(3)
for _ in range(500):
    k = rng.randint(1, 4)
    h = [rng.randint(0, 5) for _ in range(k * rng.randint(1, 3))]
    assert counter_answer(h, k) is oracle(h, k)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def isNStraightHand(self, hand, groupSize):
        if len(hand) % groupSize:
            return False
        cnt = Counter(hand)
        heap = list(cnt)
        heapify(heap)
        while heap:
            x = heap[0]
            for y in range(x, x + groupSize):
                if cnt[y] == 0:
                    return False
                cnt[y] -= 1
                if cnt[y] == 0:
                    if y != heap[0]:
                        return False
                    heappop(heap)
        return True


assert MySolutionAnswer().isNStraightHand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3) is True
assert MySolutionAnswer().isNStraightHand([1, 2, 3, 4, 5], 4) is False
rng = random.Random(4)
for _ in range(500):
    k = rng.randint(1, 4)
    h = [rng.randint(0, 5) for _ in range(k * rng.randint(1, 3))]
    assert MySolutionAnswer().isNStraightHand(list(h), k) is oracle(h, k)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def build_groups_answer(hand, groupSize):
    if len(hand) % groupSize:
        return None
    cnt = Counter(hand)
    groups = []
    for x in sorted(cnt):
        c = cnt[x]
        if c:
            for y in range(x, x + groupSize):
                if cnt[y] < c:
                    return None
                cnt[y] -= c
            groups += [list(range(x, x + groupSize)) for _ in range(c)]
    return groups


def valid_grouping(hand, k, groups):
    return (groups is not None and all(len(g) == k and all(g[i + 1] - g[i] == 1 for i in range(k - 1)) for g in groups)
            and sorted(v for g in groups for v in g) == sorted(hand))


g = build_groups_answer([1, 2, 3, 6, 2, 3, 4, 7, 8], 3)
print(g)
assert valid_grouping([1, 2, 3, 6, 2, 3, 4, 7, 8], 3, g) and build_groups_answer([1, 2, 3, 4, 5], 4) is None
rng = random.Random(5)
for _ in range(500):
    k = rng.randint(1, 4)
    h = [rng.randint(0, 5) for _ in range(k * rng.randint(1, 3))]
    gg = build_groups_answer(h, k)
    assert (gg is not None) is oracle(h, k)
    if gg is not None:
        assert valid_grouping(h, k, gg)
print("Bài 5 OK: nhóm trả về hợp lệ, có/không đúng như oracle trên 500 bộ bài")

[[1, 2, 3], [2, 3, 4], [6, 7, 8]]
Bài 5 OK: nhóm trả về hợp lệ, có/không đúng như oracle trên 500 bộ bài


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chia thành nhóm liên tiếp, `Counter`):**

```python
class Solution:
    def isNStraightHand(self, hand, groupSize):
        if len(hand) % groupSize:
            return False
        cnt = Counter(hand)
        for x in sorted(cnt):                 # từ nhỏ nhất: bắt buộc mở đầu nhóm
            c = cnt[x]
            if c:
                for y in range(x, x + groupSize):
                    if cnt[y] < c:
                        return False
                    cnt[y] -= c               # mở c nhóm cùng lúc
        return True
```

**Checklist trước khi submit:**

- [ ] Kiểm tra `len(hand) % groupSize` trước?
- [ ] Duyệt giá trị theo thứ tự **tăng dần** (lá nhỏ nhất mở đầu nhóm)?
- [ ] Dùng **số lượng** (`Counter`), không chỉ sự có mặt (`set`)?
- [ ] Trừ **`c`** (số nhóm mở tại `x`) ở mọi `y` trong nhóm?
- [ ] Duyệt trên `sorted(cnt)` (không sửa dict khi duyệt chính nó)?
- [ ] Giá trị tới `10^9`: dùng từ điển, không mảng đếm?
- [ ] Đã thử `[1,2,3,6,2,3,4,7,8],3` (True), `[1,2,3,4,5],4` (False), `[1,1,2,2,3,4],3` (False), `groupSize = 1`?
- [ ] Đúng tên `class Solution`, `isNStraightHand(self, hand, groupSize)`, không in debug.